# Suite DST — Distributions

A distribution of weights is over the matches of its symbols: its cases, each a weight and a predicate over the same
symbols, given by reference or inline, are listed in decreasing precedence, and a match weighs what the first case whose
predicate holds of it says. `Sample` draws a store's matches in proportion to their weights, from a random source the
caller gives.

In [ ]:
from collections import Counter

from mbse.Expressions import Expressions as E
from mbse.Expressions.Dialects.Python import Text
from mbse.Expressions.Framework import Terms
from mbse.Patterns import Constraints as C, Distributions as D, Predicates as P
from mbse.Schemas.Framework import JSON, Schemas as S, Stores
from support import Contact, book, listed, raises

APerson = {"person": Contact}
HasName = (P.OfPredicate.Builder().name("HasName").symbols(APerson).parameters(lambda p: p.name("name"))
           .requires(Text.FromFunction(lambda person, name: person.name == name)).create())
Names = (
    D.OfWeights.Builder().name("Names").symbols(APerson).decreasing(
        lambda wt: wt.weight(10).requires(lambda pred: pred.symbols(APerson).requires(HasName(pred.person, "alice"))),
        lambda wt: wt.weight(5).requires(lambda pred: pred.symbols(APerson).requires(HasName(pred.person, "ben"))),
        lambda wt: wt.weight(15).requires(lambda pred: pred.symbols(APerson).requires(HasName(pred.person, "chermon"))),
        lambda wt: wt.weight(7).requires(lambda pred: pred.symbols(APerson).requires(HasName(pred.person, "davi"))),
    )
    .create()
)

## DST-01 · A distribution is built fluently: symbols, and cases in decreasing precedence

In [ ]:
assert (Names.name, list(Names.symbols), Names.symbols["person"]) == ("Names", ["person"], Contact)
assert [case.weight for case in Names.cases] == [10.0, 5.0, 15.0, 7.0]  # floats, in the order given
assert all(case.predicate.rule.predicate is HasName for case in Names.cases)  # one predicate, by reference
assert Names.cases[0].predicate.name is None and Names.cases[0].predicate.rule.arguments[1].value == "alice"  # inline
Adult = P.OfPredicate.Builder().name("Adult").symbols(APerson).requires(E.variable("person").age.ge(18)).create()
more = D.OfWeights.Builder(Names).decreasing(lambda wt: wt.weight(1).requires(Adult)).clone()  # by reference too
assert len(more.cases) == 5 and more.cases[4].predicate is Adult and len(Names.cases) == 4
assert more.symbols == Names.symbols and more.symbols is not Names.symbols
assert D.OfWeights.Builder(more).name("More").update() is more and more.name == "More"
builder = D.OfWeights.Builder().symbols(APerson)
assert Terms.same(builder.person.data, E.variable("person").data)  # the builder gives its symbols as variables
case = D.OfCase.resolve(lambda wt: wt.weight(2).requires(Adult))
assert (case.weight, case.predicate) == (2.0, Adult) and D.OfCase.resolve(case) is case
with raises(TypeError, match="expected a case or a callable taking its builder, got 'x'"):
    D.OfCase.resolve("x")
assert D.DIALECT.base is P.DIALECT and list(D.DIALECT.kinds())[-9:] == [
    "weights", "case", "draw", "uniform", "normal", "poisson", "geometric", "categorical", "mixture"]
assert Names.schema_name() == "Patterns.OfWeights" and Names.cases[0].schema_name() == "Patterns.OfCase"

## DST-02 · Distributions are checked when asked, and when drawn from

In [ ]:
assert D.DIALECT.validate(Names, core=True) == []
Value = S.OfObject.Builder().name("Value").create()
odd = D.OfWeights.Builder().symbols({"person": Contact, "v": Value}).decreasing(
    lambda wt: wt.weight(-1).requires(HasName),
    lambda wt: wt.weight(1).requires(lambda pred: pred.symbols({"c": Contact}).requires(True))).create()
assert D.DIALECT.validate(odd) == [
    "symbol 'v' needs a named reference object schema",
    "case 0: its predicate's symbols must be the distribution's", "case 0: its predicate has parameters; apply it instead",
    "case 1: its predicate's symbols must be the distribution's", "argument 0: a case's weight must be positive, got -1.0"]
assert D.DIALECT.validate(D.OfWeights.Builder().symbols(APerson).create()) == ["a distribution needs a case"]
assert D.DIALECT.validate(D.OfWeights.Builder().symbols({"person": "Contact"}).decreasing(lambda wt: wt.weight(1).requires(
    lambda pred: pred.symbols(APerson).requires(True))).create()) == [
    "symbol 'person' needs a named reference object schema", "case 0: its predicate's symbols must be the distribution's"]
assert D.DIALECT.validate(D.OfWeights(None, (E.literal(1).data,), APerson)) == ["a distribution's arguments are cases"]
assert D.DIALECT.validate(D.OfCase(1.0, E.literal(True).data)) == ["a case's predicate must be a predicate"]
assert D.DIALECT.validate(D.OfCase(1.0, None)) == ["a case needs a predicate"]
assert D.DIALECT.validate(D.OfWeights(None, (D.OfCase(1.0, None),), APerson)) == ["argument 0: a case needs a predicate"]
with raises(ValueError, match="the distribution cannot be drawn from: a distribution needs a case"):
    D.Sample(book(), D.OfWeights.Builder().symbols(APerson).create(), Stores.PCG32(42))
assert D.check(Names) is Names

## DST-03 · A match weighs what its first holding case says

In [ ]:
store = book()
alice, ben, nobody = (store.Contact().name(n).create() for n in ("alice", "ben", "zed"))
evaluate = P.Evaluator(store)
assert [D.weight(evaluate, Names, {"person": p}) for p in (alice, ben, nobody)] == [10.0, 5.0, 0.0]
Overlapping = D.OfWeights.Builder().symbols(APerson).decreasing(
    lambda wt: wt.weight(3).requires(lambda pred: pred.symbols(APerson).requires(pred.person.has("name"))),
    lambda wt: wt.weight(2).requires(lambda pred: pred.symbols(APerson).requires(HasName(pred.person, "alice")))).create()
assert D.weight(evaluate, Overlapping, {"person": alice}) == 3.0  # the first case that holds, not the most specific
unknown = store.Contact().create()
assert D.weight(evaluate, Names, {"person": unknown}) == 0.0  # unknown is not holding

## DST-04 · Sample draws the store's matches in proportion to their weights

In [ ]:
store = book()
people = [store.Contact().name(n).create() for n in ("alice", "ben", "chermon", "davi", "zed")]
listed(store, *people)
drawn = D.Sample(store, Names, Stores.PCG32(42))
names = Counter(next(drawn)["person"].name for _ in range(3700))
assert set(names) == {"alice", "ben", "chermon", "davi"}  # zed weighs nothing
assert all(abs(names[n] - 100 * w) < 100 for n, w in (("alice", 10), ("ben", 5), ("chermon", 15), ("davi", 7))), names
again = [next(D.Sample(store, Names, Stores.PCG32(42)))["person"].name for _ in range(3)]
assert again == [again[0]] * 3  # the same seed, the same draws
first = [m["person"].name for m, _ in zip(D.Sample(store, Names, Stores.PCG32(42)), range(6))]
other = [m["person"].name for m, _ in zip(D.Sample(store, Names, Stores.PCG32(42).split("other")), range(6))]
assert first != other  # another stream, split from the same seed
nobody = book()
listed(nobody, nobody.Contact().name("zed").create())
with raises(ValueError, match="no match of the store's data has a weight"):
    next(D.Sample(nobody, Names, Stores.PCG32(1)))

## DST-05 · Distributions are data: a predicate used by several cases is written once

In [ ]:
store = C.OfStore(book())
text = JSON.ToJSON(store).Reachable(D.OfWeights.Schema, Names)
assert text.count('"name": "HasName"') == 1 and text.count('"kind": "predicate"') == 5  # HasName, and four inline
assert JSON.ToJSON(C.Builders).Reachable(D.OfWeights.Schema, Names) == text
copy = JSON.FromJSON(store).Reachable(D.OfWeights.Schema, text)
assert copy.symbols["person"] is Contact and [case.weight for case in copy.cases] == [10.0, 5.0, 15.0, 7.0]
assert len({id(case.predicate.rule.predicate) for case in copy.cases}) == 1  # still one predicate
assert all(Terms.same(a.predicate.rule, b.predicate.rule) for a, b in zip(copy.cases, Names.cases))
assert D.DIALECT.validate(copy, core=True) == []

## DST-06 · Distributions of values are terms: built, checked against the properties they draw, and data

In [ ]:
normal = D.Normal(lambda n: n.mean(70).deviation(8).rounded())
assert isinstance(normal, D.OfNormal) and normal.rounded is True and normal.mean.value == 70 and normal.deviation.value == 8
poisson = D.Poisson(lambda p: p.rate(1.5))
x = E.variable("x")
assert [D.domain(d) for d in (
    normal, D.Normal(lambda n: n.mean(1).deviation(1)), poisson, D.Geometric(lambda g: g.probability(0.5)),
    D.Uniform(lambda u: u.low(1).high(2)), D.Uniform(lambda u: u.low(1).high(2.5)), D.Uniform(lambda u: u.low(x).high(2)),
    D.Categorical(lambda c: c.option(1, "a").option(2, "b")), D.Categorical(lambda c: c.option(1, "a").option(2, 3)),
    D.Mixture(lambda m: m.option(1, normal).option(1, poisson)), E.literal(True).data, x.data)] == [
    "int", "float", "int", "int", "int", "float", None, "str", None, "int", "bool", None]
for make, name in ((D.Uniform, "a uniform"), (D.Normal, "a normal"), (D.Poisson, "a poisson"), (D.Geometric, "a geometric"),
                   (D.Categorical, "a categorical"), (D.Mixture, "a mixture")):
    with raises(TypeError, match=f"expected {name} or a callable taking its builder, got 'x'"):
        make("x")
assert D.DIALECT.validate(D.Categorical(lambda c: c)) == ["a categorical needs an option"]
assert D.DIALECT.validate(D.OfMixture((E.literal(1).data,))) == ["a mixture's arguments are options"]
Drawn = D.OfWeights.Builder().symbols(APerson).decreasing(
    lambda wt: wt.weight(1).requires(lambda pred: pred.symbols(APerson).requires(True))
    .draw(wt.person.age, D.Normal(lambda n: n.mean(1).deviation(1)))  # a float, for an int property
    .draw(wt.person.nope, 1).draw(x.age, 1).draw(E.literal(1), 2)).create()
assert D.DIALECT.validate(Drawn) == [
    "case 0: draw 0: person.age is int, but its distribution gives float", "case 0: draw 1: 'Contact' has no property 'nope'",
    "case 0: draw 2: 'x' is not one of the case's symbols", "argument 0: argument 2: property: argument 0: variable 'x' is not bound",
    "argument 0: argument 3: a draw's property must be a symbol's property, such as person.age"]
with raises(AttributeError, match="person"):
    D.OfCase.Builder().person  # the case's symbols are declared by its predicate
assert D.DIALECT.validate(D.OfCase(1.0, HasName, (E.literal(1).data,))) == ["a case's arguments after its predicate are draws"]
anyone = P.OfPredicate.Builder().symbols(APerson).requires(True).create()
assert D.DIALECT.validate(D.OfWeights(None, (D.OfCase(1.0, anyone, (E.literal(1).data,)),), APerson)) == [
    "argument 0: a case's arguments after its predicate are draws"]
Tagged = S.OfObject.Builder().name("Tagged").ref().properties(
    lambda p: p.name("tags").of(lambda t: t.as_indexed(lambda i: i.of(lambda t: t.as_native(str))))).create()
Tags = D.OfWeights.Builder().symbols({"it": Tagged}).decreasing(
    lambda wt: wt.weight(1).requires(lambda pred: pred.symbols({"it": Tagged}).requires(True)).draw(wt.it.tags, 1)).create()
assert D.DIALECT.validate(Tags) == []  # a property that is not a native: its type is not checked
Fine = D.OfWeights.Builder().name("Fine").symbols(APerson).decreasing(
    lambda wt: wt.weight(1).requires(lambda pred: pred.symbols(APerson).requires(True))
    .draw(wt.person.age, normal).draw(wt.person.name, D.Categorical(lambda c: c.option(3, "ann").option(1, "bo")))).create()
assert D.DIALECT.validate(Fine, core=True) == [] and Fine.cases[0].binds() == ("person",)
store = C.OfStore(book())
text = JSON.ToJSON(store).Reachable(D.OfWeights.Schema, Fine)
assert '"$schema": "Patterns.OfNormal"' in text and '"rounded": true' in text and '"$schema": "Patterns.OfDraw"' in text
copy = JSON.FromJSON(store).Reachable(D.OfWeights.Schema, text)
assert Terms.same(copy, Fine) and copy.cases[0].draws[0].target() == ("person", "age")